# Fraud risk analysis
Offline assessment on supplied data. Costs and review outcomes are assumptions, not observed business results.

## 1. Objective and assumptions
Score fraud probability, simulate three decisions, and minimize hypothetical cost subject to review capacity. No event timestamps or customer keys are available.

## 2. Environment and reproducibility
Seeds 42-45 control splitting and models. Execute from the repository root; raw CSV belongs in `data/raw/`.

In [1]:
from pathlib import Path
import json, pandas as pd, subprocess, sys
print('Python:', sys.version.split()[0])

Python: 3.11.9


## 3. Dataset loading
If analysis outputs are absent, this cell runs the pipeline without regenerating the open notebook.

In [2]:
if not Path('reports/results.json').exists():
    subprocess.run([sys.executable, 'scripts/run_analysis.py', '--skip-notebook'], check=True)
data = pd.read_csv('data/raw/transaction_fraud_risk_advanced_dataset.csv')
print('Shape:', data.shape, 'Fraud:', int(data.fraud.sum()))

Shape: (3000, 25) Fraud: 364


## 4. Data audit
Schema, missingness, duplicate identifiers, and partition counts are machine-readable in `reports/data_audit.json`.

In [3]:
audit = json.loads(Path('reports/data_audit.json').read_text())
print(pd.Series(audit['missing']).sort_values(ascending=False).head(12).to_string())
print(audit['partitions'])

authentication_level               289
distance_from_usual_location_km     75
merchant_risk_score                 60
login_to_transaction_minutes        60
transaction_amount                  53
ip_risk_score                       45
transaction_id                       0
day_of_week                          0
hour                                 0
failed_logins_24h                    0
account_age_days                     0
transactions_last_24h                0
{'train': {'rows': 1500, 'fraud': 182}, 'selection': {'rows': 450, 'fraud': 55}, 'calibration_fit': {'rows': 225, 'fraud': 27}, 'policy_selection': {'rows': 225, 'fraud': 27}, 'test': {'rows': 600, 'fraud': 73}}


## 5. Leakage assessment
`fraud` and transaction IDs are excluded. Chargebacks, velocity, distance, and upstream scores require point-in-time lineage proof before deployment.

## 6. Exploratory analysis
Charts use training rows for fraud patterns. Full-data quality charts are descriptive. Interaction cells may have small sample sizes.

![Class distribution](reports/figure_01.png)

![Missingness](reports/figure_02.png)

![Channel fraud rates](reports/figure_04.png)

![Device-distance interaction](reports/figure_10.png)

![Merchant amount interaction](reports/figure_11.png)

## 7. Feature engineering
Thirteen row-local features are defined in `fraud_pipeline.core.features`; no customer-history statistic is inferred from future records.

In [4]:
print(pd.read_csv('processed_dataset.csv', nrows=1).columns.tolist())

['transaction_id', 'hour', 'day_of_week', 'transaction_amount', 'distance_from_usual_location_km', 'transactions_last_1h', 'transactions_last_24h', 'account_age_days', 'failed_logins_24h', 'prior_chargebacks', 'device_age_days', 'merchant_risk_score', 'ip_risk_score', 'email_age_days', 'login_to_transaction_minutes', 'card_presence', 'channel', 'country', 'device_type', 'merchant_category', 'authentication_level', 'new_device', 'billing_address_mismatch', 'customer_tier', 'log_amount', 'amount_risk', 'device_distance', 'distance_risk', 'velocity_ratio', 'velocity_amount', 'new_account', 'young_device', 'night_transaction', 'weekend', 'auth_channel', 'login_speed', 'chargeback_amount', 'fraud']


## 8. Validation
Stratified train, model-selection, calibrator-fit, policy-selection, and final-test partitions are disjoint. They are not temporal validation.

In [5]:
print(pd.DataFrame(audit['partitions']).T.to_string())

                  rows  fraud
train             1500    182
selection          450     55
calibration_fit    225     27
policy_selection   225     27
test               600     73


## 9. Baseline modeling
Unweighted logistic regression with imputation, scaling, and one-hot encoding is the reference model.

In [6]:
comparison = pd.read_csv('reports/model_comparison.csv')
print(comparison[comparison.name.eq('logistic_plain_base')].to_string(index=False))

               name  roc_auc  average_precision    brier  log_loss
logistic_plain_base 0.660345           0.289115 0.101044  0.365075


## 10. Nonlinear models
Random forest and extra trees are compared under the same partition design.

In [7]:
print(comparison[comparison.name.str.startswith(('forest','extra_trees'))].head(6).to_string(index=False))

                        name  roc_auc  average_precision    brier  log_loss
     forest_plain_engineered 0.665547           0.254628 0.102216  0.350812
extra_trees_plain_engineered 0.624028           0.239003 0.104267  0.359496
  forest_weighted_engineered 0.669183           0.238792 0.154384  0.489762
           forest_plain_base 0.621082           0.228846 0.104434  0.359457
   extra_trees_weighted_base 0.594845           0.201591 0.154060  0.487483
      extra_trees_plain_base 0.590334           0.201251 0.105827  0.367434


## 11. Hyperparameter tuning
Logistic C is searched in three stratified training folds. These folds do not establish temporal stability.

In [8]:
print(pd.read_csv('reports/stratified_cv_tuning.csv').to_string(index=False))

 param_model__C  mean_test_score  std_test_score  split0_test_score  split1_test_score  split2_test_score
           0.03         0.221365        0.028419           0.241709           0.241210           0.181175
           0.10         0.216515        0.031554           0.250165           0.225073           0.174307
           0.30         0.210888        0.035274           0.253172           0.212667           0.166824
           1.00         0.206865        0.034737           0.247766           0.209979           0.162850
           3.00         0.205409        0.035195           0.247588           0.207205           0.161434
          10.00         0.204078        0.033994           0.245297           0.204895           0.162041


## 12. Class imbalance
Unweighted, class-weighted, and training-only undersampled logistic models are compared on untouched selection rows.

In [9]:
print(pd.read_csv('reports/imbalance_comparison.csv').to_string(index=False))

                      name  roc_auc  average_precision    brier  log_loss
       logistic_plain_base 0.660345           0.289115 0.101044  0.365075
    logistic_weighted_base 0.645109           0.266017 0.229258  0.668007
logistic_undersampled_base 0.658688           0.231206 0.125401  0.429966


## 13. Calibration
Sigmoid and isotonic mappings fit on calibration-fit rows and are compared on policy-selection rows.

![Calibration](reports/figure_14.png)

In [10]:
print(pd.read_csv('reports/calibration_comparison.csv').to_string(index=False))
print(pd.read_csv('reports/calibration_risk_bands.csv').to_string(index=False))

  method  roc_auc  average_precision    brier  log_loss  calibration_in_large  calibration_slope  expected_calibration_error
 sigmoid 0.638982           0.220790 0.104184  0.360464             -0.001676           2.566945                    0.061409
isotonic 0.624673           0.185057 0.105291  0.627370             -0.002043           0.000665                    0.029770
  risk_band  count  fraud_rate  mean_probability
(0.1, 0.15]    212    0.113208          0.118270
(0.15, 0.2]     12    0.250000          0.173826
 (0.2, 0.3]      1    0.000000          0.217985


## 14. Cost and threshold selection
Joint threshold search uses hypothetical costs; missing amounts have an explicit fallback loss and cannot be automatically approved.

![Cost by review threshold](reports/figure_15.png)

In [11]:
thresholds = pd.read_csv('threshold_cost_analysis.csv')
valid = thresholds[thresholds.capacity_ok]
print(valid.loc[valid.groupby('cost_scenario').total_expected_cost.idxmin()][['cost_scenario','review_threshold','block_threshold','total_expected_cost']].to_string(index=False))

        cost_scenario  review_threshold  block_threshold  total_expected_cost
                 base          0.124193         0.132910             2872.356
higher_block_friction          0.138239         1.000000             3342.188
    higher_fraud_loss          0.000000         0.107485             4726.984
   higher_review_cost          0.124193         0.132910             2971.356
 lower_review_success          0.124193         0.132910             3001.152


## 15. Explainability
Permutation importance and local base-model log-odds contributions are associations, not calibrated-probability or causal contributions. Fold importance is stratified, not temporal.

In [12]:
print(pd.read_csv('reports/feature_importance.csv').head(10).to_string(index=False))
print(pd.read_csv('reports/feature_importance_folds.csv').groupby('feature').ap_drop.agg(['mean','std']).sort_values('mean',ascending=False).head(10).to_string())
print(pd.read_csv('reports/local_explanations.csv').head().to_string(index=False))

                 feature  ap_drop_mean  ap_drop_sd
       prior_chargebacks      0.057747    0.007382
              new_device      0.050912    0.006316
       merchant_category      0.049698    0.010903
    transactions_last_1h      0.042090    0.019408
                 channel      0.027903    0.015989
     merchant_risk_score      0.027493    0.015371
           customer_tier      0.026298    0.002043
billing_address_mismatch      0.020911    0.020498
    authentication_level      0.017837    0.008893
                 country      0.016687    0.009918
                                  mean       std
feature                                         
merchant_risk_score           0.034188  0.022720
new_device                    0.028308  0.030027
billing_address_mismatch      0.024610  0.014826
ip_risk_score                 0.018813  0.005630
channel                       0.012791  0.006002
prior_chargebacks             0.008081  0.007732
transaction_amount            0.003397  0.00182

## 16. Fraud operations
Review success and financial impacts are simulated; compare approve-all, 0.50 binary, review-only, and the selected policy.

In [13]:
results = json.loads(Path('reports/results.json').read_text())
print({k:round(results[k]['cost_per_transaction'],3) for k in ['approve_all','binary_050','review_only','test_policy']})

{'approve_all': 20.505, 'binary_050': 20.505, 'review_only': 14.352, 'test_policy': 15.36}


## 17. Robustness
Controlled perturbations retain labels and do not reproduce adaptive fraud.

In [14]:
print(pd.read_csv('reports/robustness.csv')[['scenario','average_precision','cost_per_transaction','review_count','failed']].to_string(index=False))

                  scenario  average_precision  cost_per_transaction  review_count  failed
                      base           0.211013             15.359877            79       0
         amount_plus_50pct           0.211503             13.800490            77       0
           velocity_plus_2           0.211013             14.774963           128       0
      missing_amount_10pct           0.199014             15.657393            70       0
            unseen_channel           0.196358             15.372713            94       0
     ecommerce_channel_mix           0.196358             15.835543           109       0
       extreme_top_amounts           0.223820             13.638793            80       0
fraud_prevalence_weight_2x           0.344523             26.631127            82       0


## 18. Final test
The final partition was evaluated after model, calibration, and policy selection.

![Precision-recall curve](reports/figure_13.png)

In [15]:
print(json.dumps(results['test_metrics'], indent=2))
print(json.dumps(results['test_policy'], indent=2))

{
  "roc_auc": 0.6514777364768267,
  "average_precision": 0.21101291795049917,
  "brier": 0.10527934587963679,
  "log_loss": 0.3632155847205391,
  "calibration_in_large": 0.0013672538443398469,
  "calibration_slope": 3.2972579347421753,
  "expected_calibration_error": 0.048763866746419304,
  "precision_at_review": 0.19883040935672514,
  "recall_at_review": 0.4657534246575342,
  "f1_at_review": 0.2786885245901639,
  "confusion_at_review": [
    [
      390,
      137
    ],
    [
      39,
      34
    ]
  ]
}
{
  "review_threshold": 0.12419255718132183,
  "block_threshold": 0.13290953546649942,
  "total_expected_cost": 9215.926,
  "cost_per_transaction": 15.359876666666667,
  "review_count": 79,
  "block_count": 92,
  "approve_count": 429,
  "legitimate_block_count": 71,
  "false_positive_count": 137,
  "fraud_approved_count": 39,
  "fraud_capture_rate": 0.4301369863013698,
  "precision": 0.19883040935672514,
  "recall": 0.4657534246575342,
  "review_capacity_utilization": 0.8777777777

## 19. Monitoring and governance
Monitor missingness, unseen categories, score and decision rates, review capacity, mature-label precision, calibration, and realized loss. Human approval and rollback procedures are required before live use.

## 20. Conclusions and limits
The dataset lacks event and label timestamps, customer keys, feature lineage, actual review outcomes, verified costs, and a fairness audit. No future-period or production-performance claim is supported.